In [1]:

import pandas as pd
import numpy as np


def title(text):
    """Print a section heading."""
    print("\n" + "=" * 55)
    print(text)
    print("=" * 55)


title("1. INSTALLATION CHECK")
print("Pandas version:", pd.__version__)          # confirms pandas is installed


# 2. PANDAS SERIES (one column of labeled data)

title("2. PANDAS SERIES")

marks = pd.Series([80, 90, 70, 85])               # Series from a list (default index 0,1,2,3)
print(marks)

s = pd.Series([80, 90, 70], index=["Ali", "Sara", "Ahmed"])   # custom labels as index
print("\nSeries with custom index:\n", s)

print("s['Sara']:", s["Sara"])                    # access by label
print("s.iloc[0]:", s.iloc[0])                    # access by position
print("Mean:", s.mean(), "| Max:", s.max())       # Series has built-in statistics
print("Marks above 75:\n", s[s > 75])             # filter with a condition

d = {"Math": 90, "English": 75}
print("\nSeries from dictionary:\n", pd.Series(d))  # keys become the index


# 3. PANDAS DATAFRAME (a table with rows and columns)

title("3. PANDAS DATAFRAME")

df = pd.DataFrame({
    "Name": ["Ali", "Sara", "Ahmed", "Zara"],
    "Age": [21, 22, 20, 23],
    "Marks": [80, 90, 70, 85]
})
print(df)

print("\nColumn 'Name':", df["Name"].tolist())    # select one column (returns a Series)
print("Two columns:\n", df[["Name", "Marks"]])    # select multiple columns (use double brackets)
print("First row (iloc):\n", df.iloc[0])          # select a row by position
print("Row with label 1 (loc):\n", df.loc[1])     # select a row by label

df["Grade"] = ["A", "A+", "B", "A"]               # add a new column
df["Marks_Percent"] = df["Marks"] / 100 * 100     # new column from a calculation
print("\nAfter adding columns:\n", df)

print("\nMarks > 80:\n", df[df["Marks"] > 80])    # filter rows by condition
df = df.drop(columns=["Marks_Percent"])           # remove a column
print("\nShape (rows, cols):", df.shape)          # size of the DataFrame



# 4. OPENING / READING A CSV FILE
title("4. READING A CSV FILE")

# First, create a small messy CSV file so this program runs on any computer.
# (In real work you already have a file, e.g. pd.read_csv("students.csv"))
csv_text = """Name,Age,City,Study_Hours,Marks,Attendance
Ali,21,Lahore,5,78,90
Sara,22, karachi ,7,92,95
Ahmed,20,Multan,3,55,70
Zara,,Lahore,6,85,88
Bilal,23,Karachi,2,48,60
Hina,22,LAHORE,8,95,98
Hina,22,LAHORE,8,95,98
Omar,21,Multan,4,,75
Ayesha,20,Karachi,6,82,
Usman,22,Lahore,1,500,50
"""
with open("students.csv", "w") as f:               # write the text into a file
    f.write(csv_text)

df = pd.read_csv("students.csv")                  # open the CSV file into a DataFrame
print(df)

# Useful read_csv options (examples, not run here):
#   pd.read_csv("file.csv", usecols=["Name", "Marks"])   # read only some columns
#   pd.read_csv("file.csv", nrows=100)                   # read only the first 100 rows
#   pd.read_csv("file.csv", sep=";")                     # if the separator is not a comma
#   df.to_csv("clean.csv", index=False)                  # save a DataFrame back to CSV


# 5. DATA ANALYSIS (understanding the data)
title("5. DATA ANALYSIS")

print("First 3 rows:\n", df.head(3))              # look at the top rows
print("\nLast 2 rows:\n", df.tail(2))             # look at the bottom rows
print("\nShape:", df.shape)                       # (rows, columns)
print("Columns:", df.columns.tolist())            # column names
print("\nData types:\n", df.dtypes)               # type of each column

print("\nInfo:")
df.info()                                         # non-null counts and memory usage

print("\nSummary statistics:\n", df.describe())   # count, mean, std, min, quartiles, max

print("\nMissing values per column:\n", df.isnull().sum())   # count of NaN in each column
print("Duplicate rows:", df.duplicated().sum())   # count of fully repeated rows

print("\nCity counts:\n", df["City"].value_counts())   # how many times each value appears
print("Unique cities:", df["City"].nunique())     # number of different values

print("\nAverage marks by city:\n", df.groupby("City")["Marks"].mean())  # group and aggregate
print("\nSorted by Marks (top 3):\n", df.sort_values("Marks", ascending=False).head(3))


# 6. DATA CLEANING TECHNIQUES
title("6. DATA CLEANING")

clean = df.copy()                                 # work on a copy, keep the original safe

# (a) Remove duplicate rows
clean = clean.drop_duplicates()
print("After removing duplicates:", clean.shape)

# (b) Fix text problems: extra spaces and inconsistent capital letters
clean["City"] = clean["City"].str.strip().str.title()   # ' karachi ' -> 'Karachi', 'LAHORE' -> 'Lahore'
print("Cleaned cities:", clean["City"].unique())

# (c) Handle outliers using the IQR method (Marks = 500 is impossible)
Q1 = clean["Marks"].quantile(0.25)
Q3 = clean["Marks"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(f"Marks limits: {lower:.1f} to {upper:.1f}")
print("Outliers found:\n", clean[(clean["Marks"] < lower) | (clean["Marks"] > upper)])

# Marks cannot be above 100, so treat those values as missing (NaN) and fill them next
clean.loc[clean["Marks"] > 100, "Marks"] = np.nan

# (d) Handle missing values
print("\nMissing before filling:\n", clean.isnull().sum())

clean["Age"] = clean["Age"].fillna(clean["Age"].median())              # numeric: median
clean["Marks"] = clean["Marks"].fillna(clean["Marks"].median())        # numeric: median
clean["Attendance"] = clean["Attendance"].fillna(clean["Attendance"].mean())   # numeric: mean
# For a categorical column you would use the mode:
#   clean["City"] = clean["City"].fillna(clean["City"].mode()[0])
# To drop rows with missing values instead:   clean = clean.dropna()

print("Missing after filling:\n", clean.isnull().sum())

# (e) Fix data types
clean["Age"] = clean["Age"].astype(int)           # Age was float because of NaN, make it int
clean["Marks"] = clean["Marks"].astype(int)

# (f) Rename columns and reset the index
clean = clean.rename(columns={"Study_Hours": "StudyHours"})
clean = clean.reset_index(drop=True)              # index becomes 0,1,2,... again

print("\nClean data:\n", clean)
clean.to_csv("students_clean.csv", index=False)   # save the cleaned data


# 7. DATA CORRELATION
title("7. DATA CORRELATION")

# Correlation shows how two numeric columns move together (-1 to +1)
#   +1 = move together perfectly, 0 = no relationship, -1 = move in opposite directions
corr = clean[["Age", "StudyHours", "Marks", "Attendance"]].corr()
print(corr.round(2))                              # correlation matrix of all numeric columns

print("\nStudyHours vs Marks:", round(clean["StudyHours"].corr(clean["Marks"]), 2))
print("Attendance vs Marks:", round(clean["Attendance"].corr(clean["Marks"]), 2))

# Which feature is most related to Marks?
print("\nCorrelation with Marks (sorted):\n",
      corr["Marks"].drop("Marks").sort_values(ascending=False).round(2))

# Optional: heatmap (needs seaborn and matplotlib)
#   import seaborn as sns, matplotlib.pyplot as plt
#   sns.heatmap(corr, annot=True, cmap="coolwarm")
#   plt.show()

# Remember: correlation does NOT mean causation.


1. INSTALLATION CHECK
Pandas version: 2.3.3

2. PANDAS SERIES
0    80
1    90
2    70
3    85
dtype: int64

Series with custom index:
 Ali      80
Sara     90
Ahmed    70
dtype: int64
s['Sara']: 90
s.iloc[0]: 80
Mean: 80.0 | Max: 90
Marks above 75:
 Ali     80
Sara    90
dtype: int64

Series from dictionary:
 Math       90
English    75
dtype: int64

3. PANDAS DATAFRAME
    Name  Age  Marks
0    Ali   21     80
1   Sara   22     90
2  Ahmed   20     70
3   Zara   23     85

Column 'Name': ['Ali', 'Sara', 'Ahmed', 'Zara']
Two columns:
     Name  Marks
0    Ali     80
1   Sara     90
2  Ahmed     70
3   Zara     85
First row (iloc):
 Name     Ali
Age       21
Marks     80
Name: 0, dtype: object
Row with label 1 (loc):
 Name     Sara
Age        22
Marks      90
Name: 1, dtype: object

After adding columns:
     Name  Age  Marks Grade  Marks_Percent
0    Ali   21     80     A           80.0
1   Sara   22     90    A+           90.0
2  Ahmed   20     70     B           70.0
3   Zara   23  